In [ ]:
!pip install transformers==4.40.1 peft==0.4.0
!pip install sentencepiece
!pip install accelerate
!pip install torch
!pip install peft
!pip install datasets
!pip install bitsandbytes

In [ ]:
from huggingface_hub import login

# Replace "YOUR_TOKEN" with your Hugging Face token
login("YOUR_TOKEN")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from peft import PeftModel
import torch
import torch.nn.functional as F
import json
import pandas as pd
import numpy as np

In [1]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from datetime import datetime, timedelta
import re

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STOCKS = ["AAPL", "BA", "GS", "JPM"]
labels = ["negative", "neutral", "positive"]

In [ ]:
model_name = "ProsusAI/finbert"
tokenizer_finbert = AutoTokenizer.from_pretrained(model_name)
model_finbert = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
base_model = "meta-llama/Meta-Llama-3-8B"
peft_model = "FinGPT/fingpt-mt_llama3-8b_lora"

tokenizer_fingpt = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer_fingpt.pad_token = tokenizer_fingpt.eos_token

model_fingpt = AutoModelForCausalLM.from_pretrained(base_model, trust_remote_code=True).to(device)
model_fingpt = PeftModel.from_pretrained(model_fingpt, peft_model).eval()

In [ ]:
def finbert_sentiment(subject, body):
    text = subject + " " + body if body else subject
    text = text.strip()

    inputs = tokenizer_finbert(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model_finbert(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)

    labels = ["positive", "negative", "neutral"]
    pred = torch.argmax(probs, dim=1).item()

    return {
        "finbert_label": labels[pred],
        "finbert_sentiment": {labels[i]: probs[0][i].item() for i in range(len(labels))}
    }

In [ ]:
def get_label_probs(logits, tokenizer, labels):
    """Compute normalized probabilities for sentiment labels."""
    probs = F.softmax(logits, dim=-1)

    raw_scores = {}
    for label in labels:
        ids = tokenizer(label, add_special_tokens=False).input_ids
        raw_scores[label] = float(torch.mean(probs[ids]).item())

    total = sum(raw_scores.values())
    if total > 0:
        label_probs = {k: v / total for k, v in raw_scores.items()}
    else:
        label_probs = {k: 1.0 / len(labels) for k in labels}

    return label_probs

In [ ]:
def fingpt_sentiment(subject, body, stock):
    text = subject + " " + body if body else subject
    
    prompt = f"""
    Instruction: Determine the sentiment of the following news on {stock} stock.
    Choose one from {{negative, neutral, positive}}.

    Input: {text}
    Answer:
    """

    input_ids = tokenizer_fingpt(prompt, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model_fingpt(**input_ids)
        last_logits = outputs.logits[0, -1, :]
        label_probs = get_label_probs(last_logits, tokenizer_fingpt, labels)

    predicted_label = max(label_probs, key=label_probs.get)
    
    return {
        "finbert_label": predicted_label,
        "finbert_sentiment": {
            "neutral": label_probs["neutral"],
            "positive": label_probs["positive"],
            'negative': label_probs["negative"]
        }
    }

In [ ]:
def parse_source_date(source_date: str, today: datetime = None):
    if today is None:
        today = datetime.today()
    
    source_date = source_date.strip()

    try:
        return datetime.strptime(source_date, "%b %d, %Y").date()
    except ValueError:
        pass
    
    # Relative date pattern
    match = re.match(r"(\d+)\s+(day|week|hour|month|year)s?\s+ago", source_date.lower())
    if match:
        value, unit = int(match.group(1)), match.group(2)

        if unit == "day":
            return (today - timedelta(days=value)).date()
        elif unit == "week":
            return (today - timedelta(weeks=value)).date()
        elif unit == "hour":
            return (today - timedelta(hours=value)).date()
        elif unit == "month":
            return (today - timedelta(days=30 * value)).date()  # approx
        elif unit == "year":
            return (today - timedelta(days=365 * value)).date()  # approx
    
    raise ValueError(f"Unrecognized date format: {source_date}")

In [ ]:
DATE = (2025, 9, 9)
LEN_DAY = 9

QUERY = {
    'AAPL': "(AAPL)",
    'JPM': "(JPM OR \"JPMorgan Chase\")",
    'BA': "(BA OR Boeing)",
    'GS': "(GS OR OR \"Goldman Sachs\")"
}

MX_CRAWL = 50

In [ ]:
def crawl(index_query, year, month, day, length, out_file, mx_len, stock):
    all_data = []
    if os.path.exists(out_file):
        with open(out_file, "r", encoding="utf-8") as f:
            try:
                all_data = json.load(f)
                print(f"Loaded existing records.")
            except json.JSONDecodeError:
                print("Existing JSON file is empty or broken, starting fresh.")

    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")

    driver = webdriver.Chrome(options=options)

    start = 0            
    while start < mx_len:
        url = (
            f"https://www.google.com/search?q={index_query}"
            f"&tbs=cdr:1,cd_min:{month}/{day}/{year},cd_max:{month}/{day + length - 1}/{year}"
            f"&tbm=nws&hl=en&gl=us&start={start}"
        )
        
        driver.get(url)
        driver.implicitly_wait(2)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_all_elements_located((By.XPATH, '//*[@id="rso"]//a'))
            )
        except TimeoutException:
            print(f"[STOP] No more results")
            break

        articles = driver.find_elements(By.XPATH, '//*[@id="rso"]//a')
        page_data = []

        for el in articles:
            try:
                link = el.get_attribute("href")
                if not link or any(d["link"] == link for d in all_data):
                    continue

                headline = el.find_element(By.XPATH, './/div[@role="heading"]').text.strip()

                try:
                    source_date = el.find_element(By.XPATH, './/div[4]').text.strip()
                except:
                    source_date = ""

                try:
                    summary = el.find_element(By.XPATH, './/div[3]').text.strip()
                except:
                    summary = ""

                dt = parse_source_date(source_date, today=datetime.now())
                try:
                    dt = parse_source_date(source_date, today=datetime.now())
                except Exception:
                    try:
                        dt = parse_source_date(summary, today=datetime.now())
                    except Exception:
                        try:
                            dt = datetime.strptime(source_date, "%b %Y").date().replace(day=1)
                        except Exception:
                            try:
                                dt = datetime.strptime(source_date, "%Y").date().replace(month=1, day=1)
                            except Exception:
                                continue

                fingpt_data = fingpt_sentiment(headline, summary, stock)
                finbert_data = finbert_sentiment(headline, summary)
                news_data = {
                    "headline": headline,
                    "link": link,
                    "summary": summary,
                    "date": dt.isoformat(),
                    
                }
                page_data.append({**news_data, **fingpt_data, **finbert_data})
                
            except:
                continue

        if not page_data:
            print("[WARN] No new articles found.")
            break

        all_data.extend(page_data)
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        print(f"[SAVE] items saved")

        start += len(page_data)

    driver.quit()
    print(f"Finished. Total records: {len(all_data)}")

In [ ]:
for stock in STOCKS:
    crawl(QUERY[stock], DATE[0], DATE[1], DATE[2], LEN_DAY, f"Final_Data/tmp/new_{stock}.json", MX_CRAWL, stock)

In [ ]:
for stock in STOCKS:
    file1 = f"Final_Data/tmp/new_{stock}.json"
    file2 = f"Final_Data/{stock}_merged.json"

    with open(file1, "r") as f:
        data1 = json.load(f)
    with open(file2, "r") as f:
        data2 = json.load(f)

    merged = data1 + data2

    df = pd.DataFrame(merged)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    out_file = f"{stock}_merged.json"
    with open(out_file, "w") as f:
        json.dump(df.to_dict(orient="records"), f, indent=4)

    print(f"{stock}: merged {len(data1)} + {len(data2)} → {len(df)} rows, saved {out_file}")